# 📥 Notebook 01: Thu thập và Xử lý Dữ liệu Pháp luật

Notebook này thực hiện:
1. Cài đặt dependencies
2. Clone project từ GitHub (hoặc upload)
3. Thu thập văn bản pháp luật từ vbpl.vn
4. Load ALQAC 2023 dataset
5. Tiền xử lý và chuẩn hóa văn bản

In [ ]:
# === CELL 1: Cài đặt dependencies ===
!pip install -q transformers torch accelerate bitsandbytes peft trl datasets
!pip install -q sentence-transformers chromadb FlagEmbedding ragas
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q gradio beautifulsoup4 underthesea pyyaml tqdm

In [ ]:
# === CELL 2: Setup project structure ===
import os
import sys

# Nếu chạy trên Colab, mount Google Drive để lưu data
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/vietnamese-legal-qa'
    print('Running on Google Colab')
except ImportError:
    PROJECT_DIR = '.'
    print('Running locally')

# Clone hoặc cd vào project
if not os.path.exists(PROJECT_DIR):
    # Option 1: Clone từ GitHub
    # !git clone https://github.com/YOUR_REPO/vietnamese-legal-qa.git {PROJECT_DIR}
    # Option 2: Upload zip và extract
    os.makedirs(PROJECT_DIR, exist_ok=True)

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

# Tạo thư mục data
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/alqac', exist_ok=True)
os.makedirs('data/qa_dataset', exist_ok=True)
os.makedirs('data/chroma_db', exist_ok=True)
os.makedirs('models/adapters', exist_ok=True)
os.makedirs('evaluation/reports', exist_ok=True)
os.makedirs('data/feedback', exist_ok=True)

print(f'Project directory: {PROJECT_DIR}')
print('Directory structure created!')

In [ ]:
# === CELL 3: Load config ===
from src.config.settings import Settings

settings = Settings.load('config.yaml')
print('Config loaded successfully!')
print(f"LLM models: {list(settings.llm.get('models', {}).keys())}")
print(f"Embedding models: {list(settings.embedding.get('models', {}).keys())}")

In [ ]:
# === CELL 4: Download ALQAC 2023 dataset ===
# Option 1: Download từ HuggingFace (nếu có)
# from datasets import load_dataset
# dataset = load_dataset('your-alqac-dataset')

# Option 2: Upload thủ công vào data/alqac/
# Upload file ALQAC JSON vào thư mục data/alqac/

# Option 3: Tạo sample data để test pipeline
import json

sample_qa = [
    {
        "question": "Điều kiện kết hôn theo pháp luật Việt Nam là gì?",
        "answer": "Theo Điều 8 Luật Hôn nhân và Gia đình 2014, điều kiện kết hôn gồm: Nam từ đủ 20 tuổi trở lên, nữ từ đủ 18 tuổi trở lên; Việc kết hôn do nam và nữ tự nguyện quyết định; Không bị mất năng lực hành vi dân sự; Không thuộc trường hợp cấm kết hôn theo Điều 5.",
        "context": "Điều 8. Điều kiện kết hôn. 1. Nam từ đủ 20 tuổi trở lên, nữ từ đủ 18 tuổi trở lên. 2. Việc kết hôn do nam và nữ tự nguyện quyết định.",
        "source": "52/2014/QH13",
        "category": "Hôn nhân gia đình"
    },
    {
        "question": "Thời hạn hợp đồng lao động xác định tối đa là bao lâu?",
        "answer": "Theo Điều 20 Bộ luật Lao động 2019, hợp đồng lao động xác định thời hạn có thời hạn không quá 36 tháng kể từ ngày hợp đồng có hiệu lực.",
        "context": "Điều 20. Loại hợp đồng lao động. 1. Hợp đồng lao động phải được giao kết theo một trong các loại sau đây: a) Hợp đồng lao động không xác định thời hạn; b) Hợp đồng lao động xác định thời hạn là hợp đồng mà trong đó hai bên xác định thời hạn, thời điểm chấm dứt hiệu lực của hợp đồng trong khoảng thời gian không quá 36 tháng kể từ ngày hợp đồng có hiệu lực.",
        "source": "45/2019/QH14",
        "category": "Lao động"
    },
    {
        "question": "Thủ tục thành lập công ty TNHH gồm những bước nào?",
        "answer": "Theo Luật Doanh nghiệp 2020, thủ tục thành lập công ty TNHH gồm: 1) Chuẩn bị hồ sơ đăng ký doanh nghiệp (Giấy đề nghị, Điều lệ công ty, Danh sách thành viên); 2) Nộp hồ sơ tại Phòng Đăng ký kinh doanh; 3) Nhận Giấy chứng nhận đăng ký doanh nghiệp trong 3 ngày làm việc; 4) Khắc dấu và công bố thông tin.",
        "context": "Điều 21. Hồ sơ đăng ký doanh nghiệp đối với công ty trách nhiệm hữu hạn.",
        "source": "59/2020/QH14",
        "category": "Doanh nghiệp"
    }
]

# Save sample data
with open('data/alqac/sample_qa.json', 'w', encoding='utf-8') as f:
    json.dump(sample_qa, f, ensure_ascii=False, indent=2)

print(f'Sample Q&A dataset saved: {len(sample_qa)} pairs')
print('\n⚠️ Hãy upload file ALQAC 2023 đầy đủ vào data/alqac/ để có kết quả tốt hơn!')

In [ ]:
# === CELL 5: Tạo sample corpus pháp luật ===
# Trong thực tế, bạn sẽ crawl từ vbpl.vn hoặc upload files

sample_documents = [
    {
        "title": "Bộ luật Lao động",
        "document_number": "45/2019/QH14",
        "document_type": "Luật",
        "issued_date": "2019-11-20",
        "issuing_body": "Quốc hội",
        "category": "Lao động",
        "status": "Còn hiệu lực",
        "content": """Điều 20. Loại hợp đồng lao động\n1. Hợp đồng lao động phải được giao kết theo một trong các loại sau đây:\na) Hợp đồng lao động không xác định thời hạn là hợp đồng mà trong đó hai bên không xác định thời hạn, thời điểm chấm dứt hiệu lực của hợp đồng;\nb) Hợp đồng lao động xác định thời hạn là hợp đồng mà trong đó hai bên xác định thời hạn, thời điểm chấm dứt hiệu lực của hợp đồng trong khoảng thời gian không quá 36 tháng kể từ ngày hợp đồng có hiệu lực.\n\nĐiều 35. Quyền đơn phương chấm dứt hợp đồng lao động của người lao động\n1. Người lao động có quyền đơn phương chấm dứt hợp đồng lao động nhưng phải báo trước cho người sử dụng lao động như sau:\na) Ít nhất 45 ngày nếu làm việc theo hợp đồng lao động không xác định thời hạn;\nb) Ít nhất 30 ngày nếu làm việc theo hợp đồng lao động xác định thời hạn có thời hạn từ 12 tháng đến 36 tháng;\nc) Ít nhất 03 ngày làm việc nếu làm việc theo hợp đồng lao động xác định thời hạn có thời hạn dưới 12 tháng.""",
        "source_url": "https://vbpl.vn/bo-luat-lao-dong-2019"
    },
    {
        "title": "Luật Hôn nhân và Gia đình",
        "document_number": "52/2014/QH13",
        "document_type": "Luật",
        "issued_date": "2014-06-19",
        "issuing_body": "Quốc hội",
        "category": "Hôn nhân gia đình",
        "status": "Còn hiệu lực",
        "content": """Điều 8. Điều kiện kết hôn\n1. Nam từ đủ 20 tuổi trở lên, nữ từ đủ 18 tuổi trở lên.\n2. Việc kết hôn do nam và nữ tự nguyện quyết định.\n3. Không bị mất năng lực hành vi dân sự.\n4. Việc kết hôn không thuộc một trong các trường hợp cấm kết hôn theo quy định tại các điểm a, b, c và d khoản 2 Điều 5 của Luật này.\n\nĐiều 5. Bảo vệ chế độ hôn nhân và gia đình\n2. Cấm các hành vi sau đây:\na) Kết hôn giả tạo, ly hôn giả tạo;\nb) Tảo hôn, cưỡng ép kết hôn, lừa dối kết hôn, cản trở kết hôn;\nc) Người đang có vợ, có chồng mà kết hôn hoặc chung sống như vợ chồng với người khác;\nd) Kết hôn giữa những người cùng dòng máu về trực hệ; giữa những người có họ trong phạm vi ba đời.""",
        "source_url": "https://vbpl.vn/luat-hon-nhan-gia-dinh-2014"
    }
]

# Save sample documents
for i, doc in enumerate(sample_documents):
    filepath = f'data/raw/doc_{i+1:04d}.json'
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

print(f'Sample corpus saved: {len(sample_documents)} documents')
print('\n⚠️ Hãy thêm nhiều văn bản pháp luật hơn vào data/raw/ để hệ thống hoạt động tốt!')
print('Format: JSON với các fields: title, document_number, content, issued_date, category...')

In [ ]:
# === CELL 6: Load và xử lý documents ===
from src.components.data_collector import DataCollector
from src.components.text_processor import TextProcessor

collector = DataCollector(settings)
processor = TextProcessor(settings)

# Load documents
documents = collector.load_from_files('data/raw')
print(f'\nLoaded {len(documents)} documents')

# Chunk documents
all_chunks = []
for doc in documents:
    chunks = processor.semantic_chunk(doc)
    all_chunks.extend(chunks)
    print(f'  {doc.title}: {len(chunks)} chunks')

print(f'\nTotal chunks: {len(all_chunks)}')
print(f'Avg tokens per chunk: {sum(c.token_count for c in all_chunks) / len(all_chunks):.0f}')

In [ ]:
# === CELL 7: Load ALQAC dataset ===
qa_pairs = collector.load_alqac_dataset('data/alqac')
print(f'Loaded {len(qa_pairs)} Q&A pairs')

# Preview
for i, qa in enumerate(qa_pairs[:3]):
    print(f'\n--- Q&A {i+1} ---')
    print(f'Q: {qa.question[:80]}...')
    print(f'A: {qa.answer[:80]}...')
    print(f'Category: {qa.category}')

In [ ]:
# === CELL 8: Save processed data ===
import json

# Save chunks
chunks_data = [
    {
        'id': c.id,
        'content': c.content,
        'token_count': c.token_count,
        'breadcrumb': c.breadcrumb,
        'document_number': c.document_number,
        'article_number': c.article_number,
        'category': c.category,
    }
    for c in all_chunks
]

with open('data/processed/chunks.json', 'w', encoding='utf-8') as f:
    json.dump(chunks_data, f, ensure_ascii=False, indent=2)

print(f'Saved {len(chunks_data)} chunks to data/processed/chunks.json')
print('\n✅ Data collection & preprocessing complete!')
print('Next: Run notebook 02 (or 03) for embedding & indexing')